In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# load my cleaned data
df_clean = pd.read_csv('cleaned_ai_adoption_qm.csv')

# Encode categorical features (industry) and target (ai_adoption_stage)
df_clean = pd.get_dummies(df_clean, columns=['industry'], drop_first=True)

target_col = 'ai_adoption_stage'
label_encoder = LabelEncoder()
df_clean[target_col] = label_encoder.fit_transform(df_clean[target_col])

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# splitting 80% training and 20% test as rule of thumb from lec
# will scale data after splitting to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling features for logiestic regression to remove dominance of higher valued feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# using 1000 iterations
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# training begins
log_reg.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [9]:
# Make predictions on the scaled test set
y_pred = log_reg.predict(X_test_scaled)

# Evaluate performance vs what prediction was
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Logistic Regression Accuracy: 0.9959

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.96       337
           1       1.00      0.99      0.99      1040
           2       1.00      1.00      1.00     15760
           3       1.00      1.00      1.00     12863

    accuracy                           1.00     30000
   macro avg       1.00      0.98      0.99     30000
weighted avg       1.00      1.00      1.00     30000



classes are imbalanced, am going to use a weighted balance method so the lcs doesnt ignore or misclassify these.




In [10]:
# init the new model with balanced weights
log_reg2 = LogisticRegression(
    max_iter=1000, 
    random_state=42, 
    class_weight='balanced'  # penalise mistakes on smaller classes more heavily
)

# Train the new model using the same scaled training data
log_reg2.fit(X_train_scaled, y_train)

# Make new predictions
y_pred_balanced = log_reg2.predict(X_test_scaled)

# Evaluate performance
print(f"Balanced Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_balanced):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_balanced))

Balanced Logistic Regression Accuracy: 0.9870

Classification Report:
              precision    recall  f1-score   support

           0       0.75      1.00      0.86       337
           1       0.87      1.00      0.93      1040
           2       1.00      0.99      0.99     15760
           3       0.99      0.98      0.99     12863

    accuracy                           0.99     30000
   macro avg       0.90      0.99      0.94     30000
weighted avg       0.99      0.99      0.99     30000



Lower acc, but more relevant. model is now not ignoring class 0 companies jsut because there are less of them.